In [ ]:

import copy
import time
import logging
import argparse
import sys, os
import cupy as cp
import numpy  as np

import jax 
import flax.linen as nn 
import jax.numpy as jnp
from jax.lax import scan
from jax import grad, jit, vmap
import jax.random as random
from functools import partial
rng = random.PRNGKey(2022)

from math import pi
from os.path import join
import matplotlib.pyplot as plt

from levelsetpy.utilities import *
from levelsetpy.visualization import *
from levelsetpy.grids import createGrid
from levelsetpy.dynamicalsystems import RocketSystemRel
from levelsetpy.initialconditions import shapeCylinder
from levelsetpy.spatialderivative import upwindFirstENO2
from levelsetpy.explicitintegration.integration import odeCFL2, odeCFLset
from levelsetpy.explicitintegration.dissipation import artificialDissipationGLF
from levelsetpy.explicitintegration.term import termRestrictUpdate, termLaxFriedrichs


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [2]:
129/2

64.5

In [7]:
args = Bundle(dict(silent=False, save=False, visualize=False, load_brt=False, stochastic=False, elevation=25., direction=True, azimuth=45., pause_time=.3))

In [8]:
u_bound, w_bound = 1, 1
grid_bound = 64
grid_min = jnp.array((-grid_bound, -grid_bound, -pi/2))
grid_max = jnp.array((grid_bound, grid_bound, pi/2))
resolution = 100
a = 1; g = 32; u_e = 1; u_p = -1
a_p, a_e = 1, 1

In [9]:
states = Bundle(dict(x=jnp.linspace(-grid_bound, grid_bound, resolution),
                     z=jnp.linspace(-grid_bound, grid_bound, resolution),
                     theta=jnp.linspace(-jnp.pi/2, jnp.pi/2, resolution)))
states.time = jnp.linspace(0, 1, resolution)

# make states 3D 
# states.xs = jnp.meshgrid(*[states.x, states.z, states.theta], indexing='ij', sparse=False)

system=Bundle(dict())        
system.states=states
system.xdot = [
                a * jnp.cos(states.theta) + u_e * states.x,
                a * jnp.sin(states.theta) + a + u_e*states.x- g, 
                (u_p - u_e)*jnp.ones_like(states.x)
        ]
system.hamfunc = lambda p1, p2, p3: -a * p1 * jnp.cos(states.theta) - \
                        p2 * (g - a - a*jnp.sin(states.theta)) - \
                        u * abs(p1*jnp.sin(states.x) +p3) + \
                        u * abs(p2*states.x + p3)

I want you to read the following paper available at the following local path on my drive: `$HOME/Documents/Papers/MoluxLabs/ICML26/icml26.pdf`.

The paper tries to resolve the safety of a dynamical system using Hamilton-Jacobi-Isaacs optimal nonlinear control theory. The numerical computation
of the zero level set which corresponds to the safe HJIBRT is typically computed with the `levelsetpy` toolbox available at `$HOME/Documents/ML-Control-Rob/control/levelsetpy`. In the `icml26.pdf` paper, a number of innovations is made viz.,a linearization of the viscous HJ value function equation in equation (Linear-Trans). This transformation is akin to the COle-Hopf transformation used in linearizing functional HJ equations. The transformation is the cornerstone of the expected values of the smoothed HJ solution in equation (12), the spatial gradients to the perturbed value function in equation (13) and the time derivative of the value function in equation (14). You may read the appendix II for a full proof of the derived functions. 

Here's the task:

+ Verify the proofs that Lemmas 3.1 through 3.4 are correct.

+ Read through sections 3.2 and 3.3 and evaluate if the proposed importance sampling schemes for the expectations can be used in recovering a solution to the equations (HJ-RCBRT-Visc) and (HJ-RCBRAT-Visc). 

+ If possible, work with Lekan to iteratively propose a numerical solution that is implemented in python3.11 or C++20 for recovering the safe set as is done in the paper `$HOME/Documents/Papers/MSRYears/TOMS/levtoms.pdf` and implemented inthe library and folder: `$HOME/Documents/ML-Control-Rob/control/levelsetpy`

### Iteration Guides

+ Prefer brutal honesty to half-baked and incomplete solutions.

+ Explore as much associated literature as possible, providing tex write-ups for how the icml26.pdf paper may be enhanced to make 
the science it communicates clearer to the reader.

+ Feel free to ask the reader for guides on what papers to read to thoroughly understand the problem.

+ After prescribing a circumspect theory and complete solution, develop a comprehensive and wholesome algorithm to generate a numerical solution.

+ Develop a codebase that implements the algorithm --- watch out for neat memory layouts, scalability, load and soak experiments for a proof-of-concept.

+ Be imaginative in the class of examples you propose to rigorously test the algorithm. Classical examples have included two Dubins car system in relative coordinates with one another, two rockets in relative coordinates to one another based on Dreyfus' paper, murmurations, the double integrator etc (see the paper: `$HOME/Documents/Papers/MSRYears/TOMS/levtoms.pdf` for reference). While the examples in the paper may be used for a guide, theyr should not be a bus stop for your imagination. Make examples out of moleculr biology, ai safety systems, robots manipulating objects, and coding agents trying to validate a safety metric.

+ Conduct a load and soak testing strategy

+ deterministic reproduction of race conditions in your codebase

+ and execute a good cross-platform validation strategy



  Issue 2 (Non-constant transformation coefficient — the fundamental concern): The transformation coefficient $\alpha = -\frac{2H(t;\mathbf{x},D\mathbf{v}^\delta)}{\delta|D\mathbf{v}^\delta|^2}$ depends on the solution $\mathbf{v}^\delta$ itself through both $H(t;\mathbf{x},D\mathbf{v}^\delta)$
  and $|D\mathbf{v}^\delta|^2$. The chain rule in (B.2) treats $\varphi$ as a simple composition $\varphi(v^\delta)$, but actually:

  $$\omega^\delta(t;\mathbf{x}) = \exp!\Bigl(-\frac{2}{\delta}\cdot\frac{H(t;\mathbf{x},D\mathbf{v}^\delta)}{|D\mathbf{v}^\delta|^2}\cdot v^\delta\Bigr)$$

  This depends on $v^\delta$ through three pathways: (a) the explicit $v^\delta$ factor, (b) $Dv^\delta$ in the denominator, and (c) $H(t;\mathbf{x},Dv^\delta)$ in the numerator. Computing $\omega_t^\delta$ or $\Delta\omega^\delta$ rigorously requires the full product/chain rule accounting for
  derivatives of $H/|Dv^\delta|^2$ with respect to $t$ and $\mathbf{x}$.

  In the classical Cole-Hopf transformation for Burgers' equation ($v_t + vv_x = \tfrac{\delta}{2}v_{xx}$), one uses $\omega = e^{-v/\delta}$ where the exponent coefficient $-1/\delta$ is a constant. Here, the coefficient $-2H/(\delta|Dv^\delta|^2)$ is solution-dependent, which breaks the
  exactness of the linearization.

  Verdict: The proof is valid only under the implicit assumption that $H/|Dv^\delta|^2$ is locally frozen (treated as a known, spatially-varying coefficient rather than a functional of $v^\delta$). This makes the linearization an approximate/local one, not an exact global transformation. The
  paper should state this assumption explicitly. Without it, the chain rule (B.2)–(B.3) is incomplete.

  ---
  Lemma 3.2 (Smoothed HJ Solution, equation 11/12)

  Claim: 
  
  \begin{equation}
    \mathbf{v}^\delta(t;\mathbf{x}) = -\tfrac{\delta}{2}\cdot\tfrac{|Dg|^2}{H^\delta}\log\bigg\{\tfrac{1}{(\sqrt{2\pi\delta t})^n}\int_\Omega \exp(-\tfrac{1}{2\delta t}|\mathbf{x}-\mathbf{y}|^2)\exp(-\tfrac{2}{\delta}\tfrac{H^\delta}{|Dg|^2}g(\mathbf{y})),d\mathbf{y}\bigg\}
  \end{equation}

  Verification:

  Given the heat equation $\omega_t^\delta = \tfrac{\delta}{2}\Delta\omega^\delta$ with initial data $\omega^\delta(0;\mathbf{x}) = \exp(-\tfrac{2}{\delta}\tfrac{H}{|Dg|^2}g(\mathbf{y}))$, the fundamental solution (Green's function convolution) correctly gives equation (B.8a):

  $$\omega^\delta(t;\mathbf{x}) = \frac{1}{(2\pi\delta t)^{n/2}}\int_\Omega \exp!\bigl(-\tfrac{|\mathbf{x}-\mathbf{y}|^2}{2\delta t}\bigr)\exp!\bigl(-\tfrac{2H^\delta}{\delta|Dg|^2}g(\mathbf{y})\bigr),d\mathbf{y}$$

  This is the standard heat kernel convolution. Converting to a Gaussian expectation (B.8b) is also correct.

  Then, inverting via $v^\delta = -\tfrac{\delta}{2}\tfrac{|Dv^\delta|^2}{H}\log\omega^\delta$ (from B.6/B.9):


  Issue 3 (Inconsistent gradient notation): In equation (11) of the main text, the prefactor uses $|Dg|^2$ (gradient of the terminal value), but the inversion formula (B.9) uses $|Dv^\delta|^2$ (gradient of the solution at time $t$). These are equal only at $t = 0$ where $v^\delta(0;\mathbf{x}) =
   g(0;\mathbf{x})$. For $t > 0$, $Dv^\delta \neq Dg$ in general. Similarly, $H^\delta = H(t;\mathbf{x},Dv^\delta)$ in (B.9) versus the boundary Hamiltonian evaluation.

  Furthermore, comparing equations (11) and (12): equation (11) has the prefactor $-\tfrac{\delta}{2}\cdot\tfrac{|Dg|^2}{H^\delta}$, but equation (12) has simply $-\tfrac{\delta}{2}$, dropping $\tfrac{|Dg|^2}{H^\delta}$ without justification. This is an algebraic inconsistency.

  Verdict: The structure of the proof (heat kernel → log-expectation) is correct in form. However, the replacement of $|Dv^\delta|^2$ with $|Dg|^2$ and the inconsistency between (11) and (12) are errors that need correction. If one interprets this as an approximation valid near $t = 0$ or under
  slow variation of $Dv^\delta$, it could be salvaged with appropriate error bounds.

  ---
  Lemma 3.3 (Spatial Gradient, equation 13)

  Verification: Differentiating $\omega^\delta$ from (B.8a) w.r.t. $\mathbf{x}$:

  $$D_x\exp!\bigl(-\tfrac{|\mathbf{x}-\mathbf{y}|^2}{2\delta t}\bigr) = -\frac{\mathbf{x}-\mathbf{y}}{\delta t}\exp!\bigl(-\tfrac{|\mathbf{x}-\mathbf{y}|^2}{2\delta t}\bigr)$$

  Issue 4 (Possible sign error in B.11a): Equation (B.11a) writes $D\omega^\delta$ with a factor $\tfrac{(\mathbf{x}-\mathbf{y})}{t\cdot\delta}$ (positive), but the derivative above gives $-\tfrac{(\mathbf{x}-\mathbf{y})}{\delta t}$ (negative). Equation (B.11b) then has a $-\tfrac{1}{\delta t}$
  prefactor. If (B.11a) is indeed positive, then the sign flip to (B.11b) is unjustified. More likely, (B.11a) should also carry a negative sign.

  The subsequent derivation in (B.12) computes $Dv^\delta$ from $v^\delta = -\tfrac{\delta}{2}\tfrac{|Dv^\delta|^2}{H}\log\omega^\delta$ using:

  $$Dv^\delta = -\frac{\delta}{2}\left[\frac{|Dg|^2}{H}\cdot\frac{D\omega^\delta}{\omega^\delta} + \log\omega^\delta\cdot D!\left(\frac{|Dv^\delta|^2}{H}\right)\right]$$

  The second term involves $D^2v^\delta$ (the Hessian) and $DH^\delta$ — quantities that are themselves unknown.

  Verdict: The expression is not self-contained; it defines $Dv^\delta$ implicitly in terms of its own higher derivatives. This is valid as a formal identity but has limited computational utility unless one has independent estimates of $D^2v^\delta$ and $DH$.

  ---
  Lemma 3.4 (Time Derivative, equation 14)

  Verification: The time derivative of $\omega^\delta$ from the heat equation kernel gives (B.13a–c):

  $$\omega_t^\delta = \frac{-1}{2\delta t^2},\mathbb{E}_{y\sim\mathcal{N}}!\bigl[|\mathbf{x}-\mathbf{y}|^2\exp(\cdots)\bigr]$$

  This follows from $\partial_t!\bigl(-\tfrac{|\mathbf{x}-\mathbf{y}|^2}{2\delta t}\bigr) = \tfrac{|\mathbf{x}-\mathbf{y}|^2}{2\delta t^2}$, which is correct.

  From (B.14)–(B.15), the time derivative $v_t^\delta$ also involves $D_t Dv^\delta$ and $D_t H$ — again, not self-contained.

  Verdict: Same structural issue as Lemma 3.3. The formal identity is correct modulo the frozen-coefficient assumption, but the expression is implicit.

  ---
  Summary of Proof Issues
  ┌───────┬─────────────────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │ Issue │      Location       │ Severity │                                             Description                                              │
  ├───────┼─────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ 1     │ Prop. 3.1 statement │ Medium   │ RHS should be $0$, not $H$ (typo)                                                                    │
  ├───────┼─────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ 2     │ Prop. 3.1 proof     │ High     │ Non-constant transformation coefficient; chain rule incomplete without frozen-coefficient assumption │
  ├───────┼─────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ 3     │ Lemma 3.2 (11)→(12) │ Medium   │ Prefactor $                                                                                          │
  ├───────┼─────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ 4     │ Lemma 3.3 (B.11a)   │ Low      │ Possible sign error in integral form                                                                 │
  ├───────┼─────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ 5     │ Lemmas 3.3, 3.4     │ Medium   │ Expressions not self-contained — depend on unknown higher-order derivatives                          │
  └───────┴─────────────────────┴──────────┴──────────────────────────────────────────────────────────────────────────────────────────────────────┘
  ---
  Part II: Evaluation of the Importance Sampling Schemes (Sections 3.2 & 3.3)

  Section 3.2: Relation to Backward Reachable Tubes

  The viscous RCBRT equation is correctly restated as:

  $$v_t^\delta + \min{0,, H^\delta - \tfrac{\delta}{2}\Delta v^\delta} = 0$$

  The insight that the solution can be recovered from Gaussian kernel expectations (Lemmas 3.2–3.4) is sound in principle. However:

  1. The $\min{0, \cdot}$ operator introduces a non-smooth switching at the zero level set boundary. Monte Carlo estimates of the expectations inside the $\min$ will have difficulty resolving this discontinuity accurately.
  2. For the reach-avoid tube (HJI-RCBRAT-Visc), the additional $\min$ with $g(t;\mathbf{x}) - \ell(t;\mathbf{x})$ compounds this difficulty.

  Section 3.3: Sequential Importance Sampling

  The chain decomposition $v^\delta(t;\mathbf{x}) = v^\delta(d_1;\mathbf{x}1)\prod{d=2}^T v^\delta(d;\mathbf{x}d | \mathbf{x}{1:d-1})$ (equation 16) and the corresponding proposal $q^\delta$ (equation 17) are structurally valid importance sampling constructions.

  Can these IS schemes recover solutions to (HJ-RCBRT-Visc) and (HJ-RCBRAT-Visc)?

  Yes, in principle, with significant caveats:

  1. Feasibility for low-to-moderate dimensions (n ≤ 6): The Gaussian kernel expectations can be estimated via Monte Carlo with reasonable sample sizes. The IS proposal reduces variance if designed to match the target.
  2. Challenges for high dimensions: The curse of dimensionality manifests through the exponential weight $\exp(-\tfrac{2}{\delta}\tfrac{H^\delta}{|Dg|^2}g(\mathbf{y}))$. When $\delta$ is small (approaching the inviscid limit), this weight becomes very peaked, requiring exponentially many
  samples.
  3. The frozen-coefficient problem (Issue 2 above) propagates: Since $|Dv^\delta|^2$ is approximated by $|Dg|^2$, the IS proposal will be based on an approximate target density. This introduces systematic bias, not just variance.
  4. The $\min{0,\cdot}$ in the RCBRT: The switching makes the integrand non-smooth, degrading IS efficiency near the zero level set.
  5. Temporal accumulation of errors: The sequential decomposition (16) multiplies conditional densities over $T$ time steps. Estimation errors compound multiplicatively, potentially leading to exponential error growth.

  Bottom line: The IS scheme is a promising direction for moderate-dimensional problems (3–6D) with moderate viscosity ($\delta \sim 0.01$–$0.1$). It will struggle with: (a) small $\delta$ (sharp transitions), (b) high dimensions ($n > 6$), and (c) long time horizons (large $T$). The paper should
   provide error bounds or convergence rate analysis for the IS estimator.

  ---
  Part III: Toward a Numerical Implementation

  Before diving into code, let me ask some clarifying questions.                                                                                                                                                                                                                                         


In [ ]:
class ValuesApproximator(nn.Module):
    def __init__(self, 
                    # grid, 
                    u_bound=1, # control bound
                    w_bound=1, # disturbance bound
                    a=32, # rockets acceleration
                    g=64,  # graviatational acceleration
                    delta=1e-1, # viscuous parameter
                    alpha = 1.0, # regularization term 
                    jax_rand_key=123, # random key for multivariate normal draws
                    states=None, # system states (see paper)
                    xdot=None, 
                    trange=[0, 2.5],
                    batch_size=16):
        super(ValuesApproximator, self).__init__()
        assert batch_size > 0, "Batch size must be positive"
        self.batch_size = batch_size
        self.delta = delta 
        self.alpha = alpha
        self.alpha = alpha
        self.u_bound = u_bound
        self.w_bound = w_bound

        # Dynamics
        self.states = states 
        self.state_dynamics = xdot
        self.u = lambda u: u*u_bound
        self.w = lambda w: w*w_bound
        self.ap, self.ae = a, a
        self.a = a
        self.g = g
        # set linear speeds
        self.u_e = self.u(u_bound)
        self.u_p = self.u(-u_bound)
        self.g_of_y = None 
        self.Dg = None

        # jax keys
        self.jax_rand_key = jax.random.key(key)

        self.value = self.get_value(self.states, init=True)
        self.trange = trange 

    def get_value(self, states, init=False):
        # set up init value 
        if isinstance(states, Bundle):
            value = jnp.sqrt(states.x**2 + states.z**2)
        elif isinstance(states, list):
            value = jnp.sqrt(states[0]**2 + states[2]**2)

        if init:
            # setup boundary conditions
            value = value.at[0].set(0)
            value = value.at[-1].set(0)

        return value 

    def expected_value(self, t):
        """
            This implements Lemma 3.2, eq (12)
        """
        # must be symmetric & psd
        cov = jnp.identity(self.value.shape[0], dtype=jnp.float32) 
        mean = jnp.zeros_like(self.value)

        rand_draws = jax.random.multivariate_normal(key=keyjax, mean=mean, \
                            cov=cov, shape=(self.batch_size, 3))
        std_dev = jnp.sqrt(self.delta*t/self.alpha)

        y = std_dev * rand_draws + self.value
        self.g_of_y = self.get_value(y, t)

    def integrate(self):
        small = eps*100
        t_now = self.trange[0]
        while (self.trange[1] - t_now > small*self.trange[1]):
            y0 = self.value 
            

    def hamfunc(self, t, p1, p2, p3):

        Hxp = -self.a * p1 * torch.cos(self.states.theta) - \
                    p2 * (self.g - self.a - self.a*torch.sin(self.states.theta)) - \
                    u * abs(p1*torch.sin(self.states.x) +p3) + \
                    u * abs(p2*self.states.x + p3)
        
        return Hxp

    # def terminal_value(self, y):


In [46]:
value_approx = ValuesApproximator(states=system.states, xdot=system.xdot)
# value_approx.get_value(value_approx.states)
the_value =  value_approx.value 
the_value.shape 

(100,)

In [73]:
key=123
keyjax = jax.random.key(key)
# print(f'keyjax: {keyjax}')

t = 0.7
n = the_value.shape[0]
# must be symmetric & psd
cov = jnp.identity(n)
print(f'mean: {the_value.shape}')
# print(f'cov: {cov[:10,:10]}')
rand_draws = jax.random.multivariate_normal(key=keyjax, mean=jnp.zeros_like(the_value), \
    cov=cov, shape=(100,))
print(f'random_draws: {rand_draws.shape}')

mean: (100,)
random_draws: (100, 100)


In [ ]:
from flax import nnx
import optax 

class TargetScore(nnx.Module):
    def __init__(self, x_data, rngs: nnx.Rngs):
        super().__init__()
        """
            Cost function for the target set. 
                l(t;x) = ||x^2 + z^2||_2
        """ 
        din, dmid, dout = x_data.shape[0], x_data.shape[0]//2, 1
        dmid2 = dmid//2; dmid3 = dmid2//2
        # linear layers 
        self.linear1 = nnx.Linear(dmid, dmid2, rngs=rngs)
        self.linear2 = nnx.Linear(dmid2, dmid3, rngs=rngs)
        self.linear_out = nnx.Linear(dmid3, dout, rngs=rngs)

        # auxiliary layers
        self.dropout = nnx.Dropout(0.1, rngs=rngs)
        self.bn = nnx.BatchNorm(dmid2, rngs=rngs)
        self.bn2 = nnx.BatchNorm(dmid3, rngs=rngs)

    # @nn.compact
    def __call__(self, x_data):
        x = nnx.relu(self.dropout(self.bn(self.linear1(x_data))))
        x = nnx.relu(self.dropout(self.bn2(self.linear2(x))))
        return self.linear_out(x)

expand = lambda x: jnp.expand_dims(x, axis=1)
x_data = jnp.concatenate([expand(states.x), expand(states.z), expand(states.theta)], axis=1)
x_data.shape      

(100, 3)

In [ ]:
model = TargetScore(x_data, rngs=nnx.Rngs(123))
optimizer = nnx.Optimizer(model, optax.adam(learning_rate=0.001), wrt=nnx.Param)
batch_sz = 16 


def loss_func(params, model, rng, batch, delta=1e-1):
    """
    params: the current weights of the model
    model: the score function
    rng: random number generator from jax
    batch: a batch of samples from the training data, representing samples from \mu_text{data}, shape (J, N)
    
    returns an random (MC) approximation to the loss \bar{L} explained above
    """
    # payoff = jnp.linalg.norm(batch[:,0]**2 + batch[:,1]**2, 2)
    payoff = jnp.sqrt(batch[:,0]**2 + batch[:,1]**2, 2)

    value_perturb = 
    return payoff

# rng = nnx.Rngs(123)
# params = model.init(x_data, rng)['params']

@nnx.jit  # Automatic state management for JAX transforms.
def train_step(model, optimizer, x, y):
  def loss_fn(model):
    y_pred = model(x)  # call methods directly
    return ((y_pred - y) ** 2).mean()

  loss, grads = nnx.value_and_grad(loss_fn)(model)
  optimizer.update(model, grads)  # in-place updates

  return loss

In [ ]:

from levelsetpy.explicitintegration.term.term_lax_friedrich import termLaxFriedrichs
from levelsetpy.explicitintegration.term.term_restrict_update import termRestrictUpdate
from levelsetpy.utilities import eps, cputime

class GaussianSampling:
    def __init__(self, delta=1e-1, int_samples=100, alpha=1.0, linesearch_iters=0, device='cpu',
                    u_bound=1, w_bound = 1, grid_bound = 100, 
                    resolution = 100, g = 32, u_e = 1, t_range = [0., 1.],
                    u_p = -1, a= 1, u = 1):
        """ Two rockets on a plane"""
        self.delta = delta
        self.int_samples = int_samples
        self.alpha = alpha
        self.linesearch_iters = linesearch_iters

        # constants
        self.a = a
        self.g = g
        self.u = u
        self.t_range = t_range

        # gradients flag
        self.grad_compute = False

        self.sample_size = int_samples

        grid_min = torch.tensor((-grid_bound, -grid_bound, -pi/2)), 
        grid_max = torch.tensor((grid_bound, grid_bound, pi/2)),

        self._init( grid_bound, resolution, g, a, u)

    def _init(self, grid_bound, resolution, g, a, u):
        states = Bundle(dict(x=torch.linspace(-grid_bound, grid_bound, resolution, dtype=torch.float64),
                        z=torch.linspace(-grid_bound, grid_bound, resolution, dtype=torch.float64),
                        theta=torch.linspace(-torch.pi/2, torch.pi/2, resolution, dtype=torch.float64)))
        states.time = torch.linspace(0, 1, resolution, dtype=torch.float64)

        # add Dirichlet boundary conditions
        states.x[0] = 0; states.x[-1] = 0
        states.z[0] = 0; states.z[-1] = 0
        states.theta[0] = 0; states.theta[-1] = 0
        # make states 3D 
        states.xs = torch.meshgrid(*[states.x, states.z, states.theta], indexing='ij')

        system=Bundle(dict())  
        system.states = states
        system.xdot = [
                        a * torch.cos(states.theta) + u_e * states.x,
                        a * torch.sin(states.theta) + a + u_e*states.x- g, 
                        (u_p - u_e)*torch.ones_like(states.x)
                ]

        p1, p2, p3 = torch.ones_like(states.x), torch.ones_like(states.z), torch.ones_like(states.theta)
        system.hamfunc =  -a * p1 * torch.cos(states.theta) - \
                                p2 * (g - a - a*torch.sin(states.theta)) - \
                                u * abs(p1*torch.sin(states.x) +p3) + \
                                u * abs(p2*states.x + p3)

        system.cost = torch.sqrt(states.x**2 + states.z**2)


        self.states = states
        self.system = system

    def odeCFLIntegrate(self, value, tspan):
        """ 
        Integrate the ODE system forward in time by CFL constrained timesteps
       using a third order Total Variation Diminishing (TVD) Runge-Kutta
       (RK) scheme.  Details can be found in O&F chapter 3.

        """
        small=100*eps
        safe_int_factor = min(1.0, 1.2 * 0.5) 
        t = tspan[0]; steps = 0; startTime = cputime()
        numY = len(value)
        stepBound = np.zeros((numY), dtype=np.float64)
        ydot = [torch.zeros_like(v) for v in value]
        y = copy.copy(value)

        while(tspan[1] - t >= small * abs(tspan[1])):

            for i in range(len(value)):
                ydot[i], stepBound[i], schemeData = termLaxFriedrichs(t, y, schemeData) # todo 
                 
            deltaT = np.min(np.hstack((0.5*stepBound,  \
                                tspan[1] - t, realmax)))
            # Take the first substep.
            t1 = t + deltaT

            if isinstance(y, list): 
                y1 = [[] for _ in range(len(y))]  
                for i in range(len(y)):
                    y1[i] += (deltaT * ydot[i])
            else:
                y1 = y + deltaT * ydot[0]
            
            # - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
            # Second substep: Forward Euler from t_{n+1} to t_{n+2}.

            # Approximate the derivative.
            #   We will also check the CFL condition for gross violation.
            for i in range(numY):
                ydot[i], stepBound[i], schemeData = termLaxFriedrichs(t1, y1, schemeData)

            
        
    
    def cost(self, t, t_indx):
        """ 
        Returns the cost function \ell(x) = \|x^2 + \|z^2 + \|_2
        """
        # this_cost = torch.sqrt(self.states.x[t_indx]**2 + self.states.z[t_indx]**2) #, dim=1, p=2)

        this_cost = self.system.cost[t_indx]
        return this_cost

    def value_sample(self):
        t_indx = torch.multinomial(self.states.time, self.sample_size, replacement=False )   
        t = self.states.time[t_indx]       

        standard_dev = np.sqrt(self.delta * t / self.alpha)

        if self.grad_compute:
            val_grad_spatial = self.value_grad_spatial(t_indx, t, standard_dev)
            p1, p2, p3 = val_grad_spatial
            this_ham = self.hamfunc(p1, p2, p3, t_indx)
        else:
            p1, p2, p3 = torch.ones_like(self.states.x[t_indx]),\
                                 torch.ones_like(self.states.z[t_indx]), \
                                torch.ones_like(self.states.theta[t_indx])
            this_ham = self.hamfunc(p1, p2, p3, t_indx)
        
        inner_func = -(2.0/self.delta) * (this_ham/ )
        self.value_sample = -0.5 * self.delta *  torch.log(torch.softmax(this_ham, dim=0))

    def value_grad_spatial(self, t_indx, t, standard_dev):
        
        self.grad_compute = True

        return val_grad_spatial


    def value_grad_time(self):
        pass

    def hamfunc(self, p1, p2, p3, t_indx):
        hamfunc =  -self.a * p1 * torch.cos(self.states.theta[t_indx]) - \
                                p2 * (self.g - self.a - self.a*torch.sin(self.states.theta[t_indx])) - \
                                self.u * abs(p1*torch.sin(self.states.x[t_indx]) +p3) + \
                                self.u * abs(p2*self.states.x[t_indx] + p3)

        return hamfunc 



SyntaxError: invalid syntax (2713377506.py, line 85)

In [95]:
cell(3)

[nan, nan, nan]

In [84]:
rockets_brat = GaussianSampling(resolution=1000, int_samples=64)

states = rockets_brat.states 
sample_size = rockets_brat.int_samples
t_indx = torch.multinomial(states.time, sample_size, replacement=False )   
local_time = states.time[t_indx]
print(local_time.shape )

torch.Size([64])


In [93]:
states.x.shape 

torch.Size([1000])